In [8]:
# ===============================
# 1. 라이브러리
# ===============================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_recall_fscore_support, average_precision_score,
    fbeta_score, precision_recall_curve
)
from sklearn.metrics import make_scorer

from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


In [2]:
# ===============================
# 2. 데이터 로드
# ===============================
df = pd.read_csv("../data/insurance_policyholder_churn_synthetic.csv")

In [3]:
# ===============================
# 3. 피처 엔지니어링
#    → 피처를 버리지 않고 파생 피처를 추가로 생성
# ===============================
def feature_engineering(df):
    df = df.copy()
    num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
    num_cols = [c for c in num_cols if c not in
                ['customer_id','churn_flag','churn_probability_true']]

    # ── 비율 피처 (존재하는 컬럼만 생성) ──────────────────────────
    if 'premium_amount' in df.columns and 'claim_amount' in df.columns:
        df['claim_to_premium_ratio'] = df['claim_amount'] / (df['premium_amount'] + 1)

    if 'num_policies' in df.columns and 'tenure_months' in df.columns:
        df['policy_per_tenure'] = df['num_policies'] / (df['tenure_months'] + 1)

    # ── 구간화 (버킷) ────────────────────────────────────────────
    if 'tenure_months' in df.columns:
        df['tenure_bucket'] = pd.cut(
            df['tenure_months'],
            bins=[0,12,36,60,120,9999],
            labels=['lt1yr','1_3yr','3_5yr','5_10yr','gt10yr']
        ).astype(str)

    if 'age' in df.columns:
        df['age_bucket'] = pd.cut(
            df['age'],
            bins=[0,30,45,60,9999],
            labels=['young','mid','senior','elder']
        ).astype(str)

    # ── 활동 비활성 플래그 ────────────────────────────────────────
    if 'months_since_last_contact' in df.columns:
        df['is_long_inactive'] = (df['months_since_last_contact'] > 12).astype(int)

    # ── 숫자 피처 간 상호작용 (상위 중요 컬럼만) ─────────────────
    if len(num_cols) >= 2:
        top2 = num_cols[:2]
        df[f'{top2[0]}_x_{top2[1]}'] = df[top2[0]] * df[top2[1]]

    return df


def preprocess_data(df):
    df = feature_engineering(df)
    y = df['churn_flag']
    drop_cols = ['customer_id','as_of_date','churn_flag',
                 'churn_type','churn_probability_true']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return X, y


X, y = preprocess_data(df)
churn_ratio = (y == 0).sum() / (y == 1).sum()

print(f"데이터 크기: {X.shape}")
print(f"클래스 분포:\n{y.value_counts()}")
print(f"Churn 비율(비율): {churn_ratio:.2f}:1")

데이터 크기: (50000, 37)
클래스 분포:
churn_flag
0    34917
1    15083
Name: count, dtype: int64
Churn 비율(비율): 2.31:1


In [4]:
# ===============================
# 4. Train/Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
# ===============================
# 5. 컬럼 구분
# ===============================
numeric_features     = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()

print(f"\n수치형 피처 {len(numeric_features)}개 | 범주형 피처 {len(categorical_features)}개")



수치형 피처 31개 | 범주형 피처 6개


In [6]:
# ===============================
# 6. 전처리 파이프라인 (공통)
# ===============================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())          # LR/RF에도 공통 적용
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [9]:
# ===============================
# 7. Optuna로 LightGBM 하이퍼파라미터 최적화
#    scoring: F-beta (β=2) — recall 중시, precision 보호
# ===============================
BETA = 2
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 전처리 먼저 적용 (Optuna 탐색 속도 향상)
X_train_t = preprocess.fit_transform(X_train, y_train)
X_test_t  = preprocess.transform(X_test)


def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1000, step=100),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 255),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 0.2),
        # 클래스 불균형: 실제 비율 0.5~2.5배 범위 탐색
        "scale_pos_weight":  trial.suggest_float("scale_pos_weight",
                                                  churn_ratio * 0.5,
                                                  churn_ratio * 2.5),
        "random_state": 42,
        "verbose": -1,
    }

    scores = []
    for tr_idx, val_idx in cv.split(X_train_t, y_train):
        X_tr, X_val = X_train_t[tr_idx], X_train_t[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        clf = LGBMClassifier(**params)
        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                # early stopping으로 과적합 방지
                # (lightgbm v4+ 방식)
            ]
        )
        prob = clf.predict_proba(X_val)[:, 1]

        # ── threshold도 함께 탐색 ─────────────────────────────
        best_fb = 0
        for t in np.arange(0.20, 0.65, 0.05):
            pred = (prob >= t).astype(int)
            fb   = fbeta_score(y_val, pred, beta=BETA, zero_division=0)
            if fb > best_fb:
                best_fb = fb

        scores.append(best_fb)

    return np.mean(scores)


print("\nOptuna 하이퍼파라미터 탐색 시작 (100 trials)...")
sampler = TPESampler(seed=42)
study   = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n최적 F{BETA} (CV): {study.best_value:.4f}")
print("최적 파라미터:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")



Optuna 하이퍼파라미터 탐색 시작 (100 trials)...


Best trial: 94. Best value: 0.723984: 100%|██████████| 100/100 [13:57<00:00,  8.38s/it]


최적 F2 (CV): 0.7240
최적 파라미터:
  n_estimators: 800
  learning_rate: 0.012045877523473183
  max_depth: 3
  num_leaves: 36
  subsample: 0.69408105904074
  colsample_bytree: 0.7342523587341296
  min_child_samples: 95
  reg_alpha: 0.01928383689424082
  reg_lambda: 0.00024208724145481885
  min_split_gain: 0.05466863915331797
  scale_pos_weight: 2.3135980557824096


In [10]:
# ===============================
# 8. 최적 LightGBM 학습
# ===============================
best_lgbm = LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
best_lgbm.fit(X_train_t, y_train)

,boosting_type,'gbdt'
,num_leaves,36
,max_depth,3
,learning_rate,0.012045877523473183
,n_estimators,800
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.05466863915331797
,min_child_weight,0.001
,min_child_samples,95


In [11]:
# ===============================
# 9. ★ Stacking 앙상블
#    Base: LightGBM + XGBoost + RandomForest
#    Meta: Logistic Regression (확률 보정)
#
#    이유: 단일 모델 한계를 넘어 다양한 편향을 결합 →
#          Precision/Recall 동시 향상에 효과적
# ===============================
lgbm_base = LGBMClassifier(**study.best_params, random_state=42, verbose=-1)

xgb_base  = XGBClassifier(
    n_estimators     = study.best_params.get('n_estimators', 500),
    learning_rate    = study.best_params.get('learning_rate', 0.05),
    max_depth        = min(study.best_params.get('max_depth', 6), 8),
    subsample        = study.best_params.get('subsample', 0.8),
    colsample_bytree = study.best_params.get('colsample_bytree', 0.8),
    reg_alpha        = study.best_params.get('reg_alpha', 0.1),
    reg_lambda       = study.best_params.get('reg_lambda', 1.0),
    scale_pos_weight = churn_ratio,
    eval_metric      = 'aucpr',   # PR-AUC 기준
    random_state     = 42,
    verbosity        = 0
)

rf_base = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 10,
    min_samples_leaf = 20,
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1
)

# 메타 모델: 확률 보정된 Logistic Regression
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

stacking = StackingClassifier(
    estimators=[
        ("lgbm", lgbm_base),
        ("xgb",  xgb_base),
        ("rf",   rf_base),
    ],
    final_estimator = meta_lr,
    cv              = 5,
    stack_method    = "predict_proba",  # 확률 기반 스태킹
    passthrough     = False,
    n_jobs          = -1
)

print("\nStacking 앙상블 학습 중...")
stacking.fit(X_train_t, y_train)
print("완료!")


Stacking 앙상블 학습 중...
완료!


In [12]:
# ===============================
# 10. 확률 보정 (Calibration)
#     Stacking 출력 확률이 실제 비율과 맞도록 보정
#     → threshold 선택이 더 안정적으로
# ===============================
calibrated_model = CalibratedClassifierCV(
    stacking, cv=5, method='isotonic'
)
calibrated_model.fit(X_train_t, y_train)

y_prob = calibrated_model.predict_proba(X_test_t)[:, 1]

In [20]:
# ===============================
# 11. Threshold 탐색 (정밀 탐색)
# ===============================
MIN_RECALL    = 0.70
MIN_PRECISION = 0.50

print(f"\n===== Threshold Search =====")
print(f"기준: recall >= {MIN_RECALL}, precision >= {MIN_PRECISION}")
print(f"{'threshold':>10} | {'precision':>10} | {'recall':>8} | {'f1':>8} | {'fbeta':>8} | {'조건'}")
print("-" * 70)

best_threshold = 0.5
best_fbeta     = 0.0
threshold_results = []

for t in np.arange(0.05, 0.80, 0.01):
    y_pred_t = (y_prob >= t).astype(int)

    if y_pred_t.sum() == 0:
        continue

    p, r, f1_vals, _ = precision_recall_fscore_support(
        y_test, y_pred_t, average=None, zero_division=0
    )
    p1 = p[1] if len(p) > 1 else 0
    r1 = r[1] if len(r) > 1 else 0
    f1_ = f1_vals[1] if len(f1_vals) > 1 else 0

    denom = (BETA**2 * p1 + r1)
    fb = (1 + BETA**2) * p1 * r1 / denom if denom > 0 else 0

    meets = (r1 >= MIN_RECALL) and (p1 >= MIN_PRECISION)

    threshold_results.append({
        "threshold":      round(t, 2),
        "precision_1":    round(p1,  4),
        "recall_1":       round(r1,  4),
        "f1_1":           round(f1_, 4),
        "fbeta_1":        round(fb,  4),
        "meets_criteria": meets
    })

    if round(t * 100) % 1 == 0:   # 0.05 간격만 출력
        marker = " ◀" if meets else ""
        print(f"{t:>10.2f} | {p1:>10.3f} | {r1:>8.3f} | {f1_:>8.3f} | {fb:>8.3f}{marker}")

    if meets and fb > best_fbeta:
        best_fbeta     = fb
        best_threshold = t

print(f"\n★ 선택된 threshold: {best_threshold:.2f}  (F{BETA}={best_fbeta:.4f})")


===== Threshold Search =====
기준: recall >= 0.7, precision >= 0.5
 threshold |  precision |   recall |       f1 |    fbeta | 조건
----------------------------------------------------------------------
      0.05 |      0.320 |    0.989 |    0.483 |    0.697
      0.06 |      0.327 |    0.987 |    0.492 |    0.703
      0.07 |      0.334 |    0.981 |    0.499 |    0.707
      0.08 |      0.338 |    0.978 |    0.502 |    0.709
      0.09 |      0.348 |    0.968 |    0.512 |    0.714
      0.10 |      0.351 |    0.963 |    0.514 |    0.714
      0.11 |      0.357 |    0.957 |    0.520 |    0.716
      0.12 |      0.362 |    0.949 |    0.524 |    0.717
      0.13 |      0.380 |    0.935 |    0.540 |    0.723
      0.14 |      0.392 |    0.921 |    0.550 |    0.725
      0.15 |      0.398 |    0.913 |    0.555 |    0.726
      0.16 |      0.402 |    0.906 |    0.557 |    0.724
      0.17 |      0.406 |    0.898 |    0.559 |    0.723
      0.18 |      0.412 |    0.893 |    0.564 |    0.724
   

In [21]:
# ===============================
# 12. 최종 성능 평가
# ===============================
final_pred = (y_prob >= best_threshold).astype(int)

print("\n" + "="*50)
print("           ★ 최종 성능 평가 ★")
print("="*50)
print(classification_report(y_test, final_pred, digits=4))
print(f"ROC AUC:       {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR AUC (AP):   {average_precision_score(y_test, y_prob):.4f}")
print(f"F1 (churn):    {f1_score(y_test, final_pred):.4f}")
print(f"F{BETA} (churn): {fbeta_score(y_test, final_pred, beta=BETA):.4f}")


           ★ 최종 성능 평가 ★
              precision    recall  f1-score   support

           0     0.8520    0.7023    0.7699      6983
           1     0.5101    0.7176    0.5963      3017

    accuracy                         0.7069     10000
   macro avg     0.6811    0.7099    0.6831     10000
weighted avg     0.7488    0.7069    0.7175     10000

ROC AUC:       0.7912
PR AUC (AP):   0.6626
F1 (churn):    0.5963
F2 (churn): 0.6636


In [15]:
# ===============================
# 13. 시각화
# ===============================
thr_df = pd.DataFrame(threshold_results)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Churn Model - Threshold Analysis", fontsize=14, fontweight='bold')

# (a) Precision-Recall Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
axes[0].plot(rec_curve, prec_curve, lw=2, color='steelblue')
axes[0].axhline(MIN_PRECISION, color='red',    ls='--', alpha=0.7, label=f'min precision={MIN_PRECISION}')
axes[0].axvline(MIN_RECALL,    color='orange', ls='--', alpha=0.7, label=f'min recall={MIN_RECALL}')
axes[0].set_xlabel("Recall");  axes[0].set_ylabel("Precision")
axes[0].set_title("PR Curve");  axes[0].legend(fontsize=8);  axes[0].grid(alpha=0.3)

# (b) Threshold vs Precision / Recall / F-beta
axes[1].plot(thr_df['threshold'], thr_df['recall_1'],    label='Recall',   lw=2)
axes[1].plot(thr_df['threshold'], thr_df['precision_1'], label='Precision',lw=2)
axes[1].plot(thr_df['threshold'], thr_df['fbeta_1'],     label=f'F{BETA}', lw=2, ls='--')
axes[1].plot(thr_df['threshold'], thr_df['f1_1'],        label='F1',       lw=2, ls=':')
axes[1].axvline(best_threshold, color='red', ls=':', lw=2, label=f'Selected={best_threshold:.2f}')
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("Threshold vs Metrics"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# (c) 조건 충족 구간 강조
meets_df = thr_df[thr_df['meets_criteria']]
axes[2].scatter(meets_df['threshold'], meets_df['fbeta_1'],
                c='green', s=60, zorder=5, label='meets criteria')
axes[2].scatter(thr_df[~thr_df['meets_criteria']]['threshold'],
                thr_df[~thr_df['meets_criteria']]['fbeta_1'],
                c='lightgray', s=20, label='not meets')
axes[2].axvline(best_threshold, color='red', ls=':', lw=2, label=f'Best={best_threshold:.2f}')
axes[2].set_xlabel("Threshold"); axes[2].set_ylabel(f"F{BETA}")
axes[2].set_title(f"F{BETA} by Threshold (Green=Criteria Met)")
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("threshold_analysis_v2.png", dpi=120, bbox_inches='tight')
plt.close()
print("\n시각화 저장 완료: threshold_analysis_v2.png")


시각화 저장 완료: threshold_analysis_v2.png


In [16]:
# ===============================
# 14. Threshold 결과표
# ===============================
print("\n===== Threshold 결과표 (전체) =====")
print(thr_df.to_string(index=False))


===== Threshold 결과표 (전체) =====
 threshold  precision_1  recall_1   f1_1  fbeta_1  meets_criteria
      0.05       0.3196    0.9891 0.4830   0.6970            True
      0.06       0.3273    0.9871 0.4916   0.7035            True
      0.07       0.3345    0.9808 0.4988   0.7074            True
      0.08       0.3376    0.9778 0.5019   0.7089            True
      0.09       0.3479    0.9678 0.5118   0.7136            True
      0.10       0.3509    0.9632 0.5144   0.7140            True
      0.11       0.3569    0.9572 0.5200   0.7163            True
      0.12       0.3621    0.9486 0.5242   0.7165            True
      0.13       0.3799    0.9347 0.5402   0.7234            True
      0.14       0.3924    0.9208 0.5503   0.7254            True
      0.15       0.3984    0.9132 0.5548   0.7256            True
      0.16       0.4018    0.9062 0.5567   0.7243            True
      0.17       0.4056    0.8979 0.5588   0.7225            True
      0.18       0.4125    0.8926 0.5642   0